<!-- SPDX-FileCopyrightText: Copyright (c) 2026, NVIDIA CORPORATION & AFFILIATES. All rights reserved. -->
<!-- SPDX-License-Identifier: Apache-2.0 -->
# Second-look video analysis

Compare an ordinary CLI read with core `second_look()` through the same CLI.
This example makes **3 logical reads**: 1 baseline plus the helper's 2 reads.
The helper adds source-window context to its prompts, so its first answer is
not the unchanged single-read baseline. Both are shown separately below.

Keep `QUESTION` focused, retaining any original options and essential scene
context. Leave out shell/tool history and prior answers. The helper adds a
time cue; it does not shorten your prompt or require multiple-choice answers.

Use the CLI from this checkout with an already configured VSS deployment
([setup](../../../AGENTS.md)) and a registered file-backed sensor. The default
uses the configured model and sampling policy. Optional denser sampling is shown
for a backend that supports it; the plain OpenAI backend cannot apply those
overrides. No endpoint or credentials belong in this notebook.

For the Cosmos service tested here, add `"--use-base64"` to `common` and
`"--relook-container", "mkv"` to the helper arguments before running; these
avoid its URL-size and transcoded-MP4 decoding limits.

Read `vss vios timeline --sensor NAME` and replace the example scope.
All times below are seconds from the **source recording's start**, not clip-local
times. Both windows must fit in one recorded segment. Here, a known instant at
120 s gets 1.5 s on each side, clamped to that segment.

If the targeted window alone answers the question, make one ordinary targeted
read instead. The 2-read strategy retains a broader first answer for comparison
and fallback; it does not pass that context to the targeted read. The target
window must contain enough evidence to answer by itself.


In [ ]:
SENSOR = "dock_recording"
FIRST_START, FIRST_END = 0.0, 235.0
SOURCE_EVENT_TIME = 120.0
RELOOK_START = max(FIRST_START, SOURCE_EVENT_TIME - 1.5)
RELOOK_END = min(FIRST_END, SOURCE_EVENT_TIME + 1.5)
QUESTION = "What color is the box at 02:00 in the source recording?"
BASE_SAMPLING = []
RELOOK_SAMPLING = []
# Only enable for a sampling-capable vLLM backend whose policy permits overrides:
# BASE_SAMPLING = ["--fps", "1", "--max-frames", "64"]
# RELOOK_SAMPLING = ["--relook-fps", "20", "--relook-max-frames", "64"]


In [ ]:
import json
import subprocess
import time


def run_cli(args):
    start = time.perf_counter()
    completed = subprocess.run(["vss", "vlm", "run", *args], capture_output=True, text=True)
    elapsed = time.perf_counter() - start
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode not in (0, 6):
        raise RuntimeError(f"VSS exited {completed.returncode}; see the diagnostic above.")
    # The first JSON object is the result; a completion marker follows it.
    result, _ = json.JSONDecoder().raw_decode(completed.stdout.lstrip())
    return result, elapsed


common = [
    "--sensor", SENSOR, "--start-time", str(FIRST_START), "--end-time", str(FIRST_END),
    "--prompt", QUESTION,
] + BASE_SAMPLING
baseline, baseline_seconds = run_cli(common)
print("Ordinary CLI baseline:", baseline["answer"])
print("Wall time (s):", round(baseline_seconds, 2))


In [ ]:
result, relook_seconds = run_cli(common + [
    "--relook-start-time", str(RELOOK_START), "--relook-end-time", str(RELOOK_END),
] + RELOOK_SAMPLING)
print("Helper first answer:", result["second_look"]["first_answer"])
print("Helper narrower answer:", result["second_look"]["second_answer"])
print("Selected answer:", result["answer"])
print("Wall time for both helper reads (s):", round(relook_seconds, 2))
print(json.dumps(result["second_look"], indent=2))


The optional vLLM settings request 20 FPS in a 3 s window with a 64-frame cap,
rather than reusing 1 FPS and receiving only about 3 frames. Empty sampling lists
leave the configured policy unchanged. Verify that the backend honors any
overrides; the request alone does not establish what was decoded.

Review both answers, source windows, sampling, job IDs and `fallback_reason`.
If answers disagree, report the disagreement and the selected answer.
Selection of the narrower answer is not proof of an accuracy gain. Compare the
ordinary baseline and selected answers against labels on a fixed paired cohort.
Exit 6 retains an answer with a stated limitation; do not retry the command to
repair the optional read. Do not enable a third read inside the strategy.
